In [0]:
# ============================================================
# Pipeline : covid19_medallion_pipeline
# Notebook : 04_gold/01_dim_condition
# Purpose  : SCD Type 1 for SNOMED-CT condition codes.

# ============================================================

import dlt
from pyspark.sql import functions as F


@dlt.view(name="v_condition_changes")
def v_condition_changes():
    df = dlt.read_stream("covid19_lakehouse.silver.conditions")
    return df.select(
        F.xxhash64("condition_code").alias("condition_key"),
        "condition_code",
        "condition_description",
        "_ingested_at",
    )


dlt.create_streaming_table("covid19_lakehouse.gold.dim_condition")

dlt.apply_changes(
    target="covid19_lakehouse.gold.dim_condition",
    source="v_condition_changes",
    keys=["condition_code"],
    sequence_by=F.col("_ingested_at"),
    stored_as_scd_type=1,
)